In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n\n{device}")

/kaggle/input/candidats-summary/Resume 2 Specialist IT_summary.txt
/kaggle/input/candidats-summary/Resume 1 Specialist IT_summary.txt
/kaggle/input/candidats-summary/Resume 2 Bisunes Analitic_summary.txt
/kaggle/input/my_model/pytorch/default/7/model.safetensors.index.json
/kaggle/input/my_model/pytorch/default/7/tokenizer.model.v3
/kaggle/input/my_model/pytorch/default/7/model-00003-of-00003.safetensors
/kaggle/input/my_model/pytorch/default/7/config.json
/kaggle/input/my_model/pytorch/default/7/params.json
/kaggle/input/my_model/pytorch/default/7/README.md
/kaggle/input/my_model/pytorch/default/7/tokenizer.json
/kaggle/input/my_model/pytorch/default/7/model-00001-of-00003.safetensors
/kaggle/input/my_model/pytorch/default/7/tokenizer_config.json
/kaggle/input/my_model/pytorch/default/7/model-00002-of-00003.safetensors
/kaggle/input/my_model/pytorch/default/7/special_tokens_map.json
/kaggle/input/my_model/pytorch/default/7/.gitattributes
/kaggle/input/my_model/pytorch/default/7/tokeni

In [2]:
# Install latest bitsandbytes & transformers, accelerate from source
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
# Other requirements for the demo
!pip install gradio
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 28.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.9 MB/s eta 0:00:00:00:0100:01
  Installing b

In [3]:
!pip uninstall -y transformers peft accelerate
!pip install -q transformers==4.44.2 peft==0.11.1 accelerate==0.33.0 sentence-transformers striprtf python-docx mammoth

Found existing installation: transformers 4.57.0.dev0
Uninstalling transformers-4.57.0.dev0:
  Successfully uninstalled transformers-4.57.0.dev0
Found existing installation: peft 0.17.2.dev0
Uninstalling peft-0.17.2.dev0:
  Successfully uninstalled peft-0.17.2.dev0
Found existing installation: accelerate 1.11.0.dev0
Uninstalling accelerate-1.11.0.dev0:
  Successfully uninstalled accelerate-1.11.0.dev0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 67.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.5 MB/s eta 0:00:00:00:01


In [4]:
!pip install python-docx

In [5]:
# === Импорты ===
import docx
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [6]:
model_path="/kaggle/input/my_model/pytorch/default/7"

tokenizer = AutoTokenizer.from_pretrained(model_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
class ChatEngine:
    def __init__(self, model, tokenizer, job_description, candidate_vector, device="cuda"):
        self.model = model
        self.tokenizer = tokenizer
        self.job_description = job_description
        self.candidate_vector = candidate_vector
        self.device = device

        self.chat_history = []   # [(роль, текст)]
        self.summary = ""        # краткое резюме
        self.turns_since_summary = 0

    def build_prompt(self):
        """Формируем prompt с учётом вакансии, вектора и истории"""
        history_text = ""
        for role, text in self.chat_history[-6:]:
            history_text += f"{role.upper()}: {text}\n"

        prompt = f"""
            Ты HR-ассистент, проводишь собеседование.
            
            Описание вакансии:
            {self.job_description}
            
            Известные характеристики кандидата:
            {self.candidate_vector}
            
            Краткое содержание предыдущей беседы:
            {self.summary}
            
            История последних сообщений:
            {history_text}
            
            Задача: сгенерируй только следующий уместный вопрос кандидату на русском языке,
            ориентируясь на требования вакансии и его опыт. 
            Не пиши ответ за кандидата. 
            Выведи только вопрос HR.
        """
        
        return prompt.strip()

    def ask_model(self, prompt, max_new_tokens=150):
        """Генерация из модели"""
        messages = [
            {"role": "system", "content": "Ты — HR для проведения собеседований."},
            {"role": "user", "content": prompt}
        ]
        inputs = self.tokenizer.apply_chat_template(messages, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output = self.model.generate(
                inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.9
            )
        decoded = self.tokenizer.decode(output[0], skip_special_tokens=True)

        if "HR:" in decoded:
            decoded = decoded.split("HR:")[-1].strip()
        return decoded

    def summarize_history(self):
        """Сжимает историю диалога в краткое резюме"""
        if len(self.chat_history) < 6:
            return

        history_text = ""
        for role, text in self.chat_history:
            history_text += f"{role}: {text}\n"

        prompt = f"""
            Ты — HR ассистент.
            Сожми кратко диалог ниже, выделив ключевые навыки, опыт и эмоциональные реакции кандидата. 
            Не повторяй дословно, делай конспект.
            
            Диалог:
            {history_text}
            
            Краткое резюме:
        """
        
        summary_text = self.ask_model(prompt, max_new_tokens=120)
        self.summary = summary_text.strip()
        self.chat_history = self.chat_history[-4:]  # оставляем только последние ходы
        self.turns_since_summary = 0

    def chat(self, candidate_reply: str = None):
        """Добавляем ответ кандидата (если есть), генерируем новый вопрос HR"""
        if candidate_reply:
            self.chat_history.append(("CANDIDATE", candidate_reply))
            self.turns_since_summary += 1

        # если накопилось много реплик — сжать
        if self.turns_since_summary >= 5:
            self.summarize_history()

        prompt = self.build_prompt()
        question = self.ask_model(prompt)

        self.chat_history.append(("HR", question))
        return question

In [16]:
# === загрузка кандидата ===
"""vec_path = "/kaggle/input/candidats-vectors/Resume 2 Specialist IT.docx.npy"
candidate_vector = np.load(vec_path)
vector_text = " ".join([f"{x:.4f}" for x in candidate_vector[:50]])"""

with open("/kaggle/input/candidats-summary/Resume 2 Specialist IT_summary.txt", "r", encoding="utf-8") as file:
    candidate_vector = file.readlines() 
candidate_vector = "".join([i for i in candidate_vector[1:]])
print(candidate_vector)
# === загрузка вакансии ===
job_doc_path = "/kaggle/input/for-llama/Description of Specialist IT.docx"
doc = docx.Document(job_doc_path)
job_description = "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

Score: 0.8062

Представление о кандидате:
Специалист ЦОД Настройка и установка серверного и телекоммуникационного оборудования (сервера, СХД, Магнитно-ленточные хранилища, криптошлюзы, коммутаторы, межсетевые экраны)



In [17]:
# === инициализация ===
engine = ChatEngine(model, tokenizer, job_description, candidate_vector)

# старт интервью
engine.chat_history.append(("HR", "Здравствуйте! Давайте начнём собеседование."))
engine.chat_history.append(("CANDIDATE", "Здравствуйте! Да, я готов."))

# HR задаёт вопрос
q1 = engine.chat()
print("Первый вопрос:\n", q1)

# кандидат отвечает
q2 = engine.chat("У меня есть опыт работы с серверным оборудованием и сетями LAN.")
print("\nСледующий вопрос:\n", q2)

# ещё ответ
q3 = engine.chat("Также занимался первичной диагностикой серверов х86 и настройкой RAID.")
print("\nСледующий вопрос:\n", q3)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected 

Первый вопрос:
 Как вы реализовывали настройку и установку серверного оборудования в предыдущих проектах? Можете ли вы привести примеры из вашего опыта, где вы решали сложные проблемы с настройкой и установкой оборудования?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Следующий вопрос:
 Можете ли вы рассказать о конкретном проекте, где вы решали сложные проблемы с настройкой и установкой криптошлюз или межсетевых экранов?

Следующий вопрос:
 Можете ли вы рассказать о конкретном проекте, где вы решали сложные проблемы с настройкой и установкой магнитно-ленточных хранилищ или коммутаторов?
